# Неробастный и робастный t-тест: значимость коэффициентов

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_params, summary_col # вывод результатов тестирования
from scipy.stats import t # t-распределение и критические значения

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

<div style="background-color:Bisque; color:DarkBlue; padding:30px;">

<i><b><span style="color: purple">Неробастный и робастный t-тест: значимость коэффициента</span></b></i><br>

Тестируем нулевую гипотезу $H_0:\beta=0$

Альтернативная гипотеза $H_1:\beta\ne 0$

Тестовая статистика 
$$t=\frac{\hat{\beta}}{s.e.(\beta)}$$

Стандартная ошибка $s.e.(\beta)$ — корень из диагонального элемента выбранной оценки ковариационной матрицы коэффициентов (далее CME = Covariance Matrix Estimator):

* неробастной $Var_{OLS}(\hat{\beta})=s^2(X^\top X)^{-1}$, где $s^2=\sum e^2_i/(n-k-1)$
* робастной к гетероскедастичности (HC3-оценки)

Оценка коэффициента $\hat{\beta}$ в обоих вариантах одна и та же, меняется только стандартная ошибка, поэтому меняются t-статистика и P-значение

При справедливости $H_0$ статистика сравнивается с t-распределением с $df=n-k-1$ степенями свободы (в коде робастной подгонки — `use_t=True`)

Критическое значение $t_{cr}=t_{df}(\alpha)$ — двустороннее: $P(|t_{df}|>t_{cr})=\alpha$; в Python — `t.isf(q=sign_level/2, df=mod.df_resid)`

$\color{purple}{\textbf{Значим ли коэффициент }\beta ?}$

**<span style="color:purple">Способ 1:</span>** используем t-статистику

* Отвергаем гипотезу $H_0$ при $|t|>t_{cr}$, <i><b><span style="color: purple">коэффициент значим</span></b></i><br>
* Не отвергаем гипотезу $H_0$ при $|t|<t_{cr}$, <i><b><span style="color: purple">коэффициент незначим</span></b></i><br>
**<span style="color:purple">Способ 2:</span>** используем P-значение

* Отвергаем гипотезу $H_0$ при $P<\alpha$, <i><b><span style="color: purple">коэффициент значим</span></b></i><br>
* Не отвергаем гипотезу $H_0$ при $P>\alpha$, <i><b><span style="color: purple">коэффициент незначим</span></b></i><br>

---

<i><b><span style="color: purple">Когда нужен робастный t-тест?</span></b></i><br>

Неробастные стандартные ошибки корректны при гомоскедастичности; при гетероскедастичности они смещены, и вывод о значимости может быть неверным. Робастные (HC3) корректны в обоих случаях. Если выводы двух тестов расходятся, следует опираться на робастный

---

</div>

In [ ]:
# импорт данных
df = pd.read_csv('../datasets/Labour.csv')

Рассмотрим регрессию

$$
\log(output)=\beta_0+\beta_1\log(capital)+\beta_2\log(labour)+\beta_3\log^2(capital)+\beta_4\log^2(labour)+error
$$

## Спецификация и подгонка

Подгоним модель методом OLS с двумя опциями
* с оценкой ковариационной матрицы коэффициентов по умолчанию (неробастной)
* с оценкой ковариационной матрицы коэффициентов, робастной к гетероскедастичности (HC3-оценкой)

In [ ]:
# спецификация модели
mod = ols(formula='np.log(output)~np.log(capital)+np.log(labour)+I(np.log(capital)**2)+I(np.log(labour)**2)', data=df)
# подгонка модели с ковариационной матрицей по умолчанию (неробастной)
res_ols = mod.fit()
# подгонка модели с робастной HC3 ковариационной матрицей
res_hc = mod.fit(cov_type='HC3', use_t=True)

## Гипотеза

Тестируется гипотеза $H_0:\beta_{j}=0, \quad (j=1, \dots, 4)$

Пусть уровень значимости $\alpha=5\%=0.05$

Критическое значение t-распределения

In [ ]:
# уровень значимости
sign_level = 0.05
# критическое значение t-распределения
t.isf(q=sign_level/2, df=mod.df_resid)

## Результаты неробастного t-теста на значимость коэффициентов

In [ ]:
# составляющие неробастного t-теста в одной таблице
summary_params(res_ols, alpha=sign_level).round(3)

**ВЫВОД**: на уровне значимости 5% значимы коэффициенты при регрессорах:
* $\log(capital)$
* $\log(labour)$
* $\log^2(capital)$
* $\log^2(labour)$

## Результаты робастного t-теста на значимость коэффициентов (HC3-оценка ковариационной матрицы)

In [ ]:
# составляющие робастного t-теста в одной таблице
summary_params(res_hc, alpha=sign_level).round(3)

**ВЫВОД**: на уровне значимости 5% значимы коэффициенты при регрессорах:
* $\log(capital)$
* $\log(labour)$
* $\log^2(capital)$

## Результаты двух оцениваний в одной таблице

__Обратим внимание на коды значимости (количество звёздочек у коэффициентов)__

In [ ]:
# результаты двух оцениваний в одной таблице
summary_col([res_ols, res_hc], model_names=['OLS-CME', 'HC3-CME'], stars=True)